
# GVH Nested Diagonal Geometry Diagnostic 0.1

**Statut :** notebook exploratoire de fondation géométrique  
**Branche recommandée :** `0.1_foundations/geometry_research/`  
**Compagnon conceptuel :** `GVH_Nested_Diagonal_Geometry_Diagnostic_Draft_0.1.md`

## Verrou unique

Ce notebook teste uniquement la cohérence numérique des diagnostics

\[
D_n,\qquad d_n,\qquad \ell_n,\qquad
\xi_n,\qquad \rho_n,\qquad \eta_n
\]

dans une géométrie euclidienne et pour des domaines cubiques imbriqués.

Il **ne constitue pas** une loi dynamique GVH.

Il ne teste pas :

- l'action GVH ;
- les contraintes ADM/Dirac ;
- le symbole physique \(A_{\rm phys}\) ;
- les ghosts ;
- les observables astrophysiques ;
- les métriques courbes.

Le front canonique principal `.28.21.2.1.1` reste séparé.


In [1]:

from __future__ import annotations

import sys, json, math
from pathlib import Path

import numpy as np
import pandas as pd

print("Python =", sys.version.split()[0])
print("NumPy =", np.__version__)
print("Pandas =", pd.__version__)

GVH_NDG_VERSION = "0.1"
GVH_NDG_STATUS = "EXPLORATORY_GEOMETRIC_DIAGNOSTIC"


Python = 3.13.15
NumPy = 2.1.3
Pandas = 2.2.3



# 1. Définitions

Pour un cube de côté \(a_n\),

\[
\boxed{
D_n=\sqrt3\,a_n.
}
\]

Pour une courbe régulière \(\mathbf r_n(t)\),

\[
\boxed{
\ell_n
=
\int_{t_0}^{t_1}
\|\mathbf r_n'(t)\|\,dt.
}
\]

Pour ses extrémités physiques,

\[
\boxed{
d_n=
\|\mathbf r_n(t_1)-\mathbf r_n(t_0)\|.
}
\]

Les trois diagnostics sans dimension sont :

\[
\boxed{
\xi_n=\frac{\ell_n}{d_n}
}
\]

\[
\boxed{
\rho_n=\frac{d_n}{D_n}
}
\]

\[
\boxed{
\eta_n=\frac{\ell_n}{D_n}
=\xi_n\rho_n.
}
\]

Lorsque les extrémités sont deux sommets opposés,

\[
d_n=D_n,
\]

donc

\[
\rho_n=1,
\qquad
\eta_n=\xi_n\ge1.
\]


In [2]:

SQRT3 = np.sqrt(3.0)

def cube_diagonal(a: float) -> float:
    if a <= 0:
        raise ValueError("Cube side a must be positive.")
    return SQRT3 * float(a)

def arc_length_from_samples(points: np.ndarray) -> float:
    points = np.asarray(points, dtype=float)
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError("points must have shape (N,3)")
    return float(np.linalg.norm(np.diff(points, axis=0), axis=1).sum())

def direct_distance(points: np.ndarray) -> float:
    points = np.asarray(points, dtype=float)
    return float(np.linalg.norm(points[-1] - points[0]))

def diagnostics_from_points(points: np.ndarray, a: float) -> dict:
    ell = arc_length_from_samples(points)
    d = direct_distance(points)
    D = cube_diagonal(a)

    xi = np.nan if d == 0 else ell / d
    rho = d / D
    eta = ell / D

    return {
        "a": float(a),
        "D": D,
        "d": d,
        "ell": ell,
        "xi": xi,
        "rho": rho,
        "eta": eta,
        "factorization_error": eta - xi * rho if d != 0 else np.nan,
    }

def inside_cube(points: np.ndarray, a: float, tol: float=1e-12) -> bool:
    points = np.asarray(points, dtype=float)
    return bool(
        np.all(points >= -tol)
        and np.all(points <= a + tol)
    )



# 2. Familles de trajectoires de référence

On teste quatre classes reliant les sommets opposés :

1. diagonale droite ;
2. courbe légèrement déviée ;
3. hélice effilée autour de la diagonale ;
4. trajet sur trois arêtes.

Pour les trois premières familles, les courbes sont échantillonnées finement.

Le trajet sur trois arêtes est calculé exactement comme polyligne.


In [3]:

e1 = np.array([1.0, -1.0, 0.0]) / np.sqrt(2.0)
e2 = np.array([1.0, 1.0, -2.0]) / np.sqrt(6.0)

def straight_curve(a: float, t: np.ndarray) -> np.ndarray:
    t = np.asarray(t)
    return a * np.column_stack([t, t, t])

def bowed_curve(a: float, t: np.ndarray) -> np.ndarray:
    t = np.asarray(t)
    amp = 0.12 * a
    env = 4.0 * t * (1.0 - t)
    wiggle = amp * env * np.sin(np.pi * t)
    return a * np.column_stack([t,t,t]) + wiggle[:,None] * e1[None,:]

def tapered_helix(a: float, t: np.ndarray, turns: int=2) -> np.ndarray:
    t = np.asarray(t)
    amp = 0.07 * a
    env = np.sin(np.pi * t)
    phase = 2.0 * np.pi * turns * t
    transverse = (
        np.cos(phase)[:,None] * e1[None,:]
        + np.sin(phase)[:,None] * e2[None,:]
    )
    return a * np.column_stack([t,t,t]) + (amp*env)[:,None] * transverse

def three_edges(a: float) -> np.ndarray:
    return np.array([
        [0.0,0.0,0.0],
        [a,0.0,0.0],
        [a,a,0.0],
        [a,a,a],
    ], dtype=float)

t_dense = np.linspace(0.0, 1.0, 20001)

path_builders = {
    "straight": lambda a: straight_curve(a, t_dense),
    "bowed": lambda a: bowed_curve(a, t_dense),
    "tapered_helix": lambda a: tapered_helix(a, t_dense),
    "three_edges": three_edges,
}



# 3. Test A — borne diagonale et décomposition \(\eta=\xi\rho\)

Pour chaque trajectoire reliant les sommets opposés, on demande :

\[
d=D,
\qquad
\xi\ge1,
\qquad
\rho=1,
\qquad
\eta=\xi\rho.
\]

Le test est répété pour plusieurs échelles imbriquées.


In [4]:

cube_sides = [1.0, 2.0, 4.0, 8.0]

rows = []

for a in cube_sides:
    for name, builder in path_builders.items():
        pts = builder(a)
        diag = diagnostics_from_points(pts, a)

        rows.append({
            "level_side_a": a,
            "path": name,
            "inside_cube": inside_cube(pts, a),
            **diag,
        })

diag_df = pd.DataFrame(rows)

TOL = 5e-5  # discretization tolerance for sampled curved paths

GNDG01_DIAGONAL_BOUND_PASS = bool(
    np.all(diag_df["inside_cube"])
    and np.all(diag_df["xi"] >= 1.0 - TOL)
    and np.all(np.abs(diag_df["rho"] - 1.0) < TOL)
    and np.all(np.abs(diag_df["factorization_error"]) < 1e-12)
)

assert GNDG01_DIAGONAL_BOUND_PASS

print(diag_df.to_string(index=False))
print("GNDG01_DIAGONAL_BOUND_PASS =", GNDG01_DIAGONAL_BOUND_PASS)


 level_side_a          path  inside_cube   a         D         d       ell       xi  rho      eta  factorization_error
          1.0      straight         True 1.0  1.732051  1.732051  1.732051 1.000000  1.0 1.000000                  0.0
          1.0         bowed         True 1.0  1.732051  1.732051  1.752219 1.011644  1.0 1.011644                  0.0
          1.0 tapered_helix         True 1.0  1.732051  1.732051  1.845599 1.065557  1.0 1.065557                  0.0
          1.0   three_edges         True 1.0  1.732051  1.732051  3.000000 1.732051  1.0 1.732051                  0.0
          2.0      straight         True 2.0  3.464102  3.464102  3.464102 1.000000  1.0 1.000000                  0.0
          2.0         bowed         True 2.0  3.464102  3.464102  3.504437 1.011644  1.0 1.011644                  0.0
          2.0 tapered_helix         True 2.0  3.464102  3.464102  3.691199 1.065557  1.0 1.065557                  0.0
          2.0   three_edges         True 2.0  3.


# 4. Test B — invariance sous reparamétrisation

Pour une même courbe géométrique, on compare plusieurs paramétrisations monotones :

\[
t=s,
\qquad
t=s^2,
\qquad
t=s^3.
\]

La longueur d'arc doit être indépendante du paramétrage dans la limite numérique.

On teste la courbe hélicoïdale effilée.


In [5]:

def sample_reparam_curve(a: float, power: int, n: int=30001) -> np.ndarray:
    s = np.linspace(0.0, 1.0, n)
    t = s**power
    return tapered_helix(a, t)

reparam_rows = []
a = 1.0

for power in [1,2,3,5]:
    pts = sample_reparam_curve(a, power)
    d = diagnostics_from_points(pts, a)
    reparam_rows.append({
        "t=s^p power": power,
        "ell": d["ell"],
        "xi": d["xi"],
        "rho": d["rho"],
        "eta": d["eta"],
    })

reparam_df = pd.DataFrame(reparam_rows)
base = reparam_df.iloc[0]

max_rel_ell = float(np.max(np.abs(reparam_df["ell"] - base["ell"])) / base["ell"])
max_rel_eta = float(np.max(np.abs(reparam_df["eta"] - base["eta"])) / base["eta"])

GNDG01_REPARAMETERIZATION_PASS = (
    max_rel_ell < 2e-6
    and max_rel_eta < 2e-6
)

assert GNDG01_REPARAMETERIZATION_PASS

print(reparam_df.to_string(index=False))
print("max relative ell difference =", max_rel_ell)
print("max relative eta difference =", max_rel_eta)
print("GNDG01_REPARAMETERIZATION_PASS =", GNDG01_REPARAMETERIZATION_PASS)


 t=s^p power      ell       xi  rho      eta
           1 1.845599 1.065557  1.0 1.065557
           2 1.845599 1.065557  1.0 1.065557
           3 1.845599 1.065557  1.0 1.065557
           5 1.845599 1.065557  1.0 1.065557
max relative ell difference = 9.120242377366793e-09
max relative eta difference = 9.120242480268928e-09
GNDG01_REPARAMETERIZATION_PASS = True



# 5. Test C — autosimilarité imbriquée

On considère :

\[
a_n=2^n a_0
\]

et une même forme de trajectoire agrandie avec le cube :

\[
\mathbf r_n(t)=2^n\mathbf r_0(t).
\]

Alors :

\[
\ell_n\propto a_n,
\qquad
d_n\propto a_n,
\qquad
D_n\propto a_n.
\]

Le diagnostic prédit :

\[
\boxed{
\xi_n,\rho_n,\eta_n
\text{ constants.}
}
\]


In [6]:

autosim_rows = []

for n, a in enumerate(cube_sides):
    pts = tapered_helix(a, t_dense)
    d = diagnostics_from_points(pts, a)

    autosim_rows.append({
        "n": n,
        "a_n": a,
        "D_n": d["D"],
        "ell_n": d["ell"],
        "d_n": d["d"],
        "xi_n": d["xi"],
        "rho_n": d["rho"],
        "eta_n": d["eta"],
    })

autosim_df = pd.DataFrame(autosim_rows)

for col in ["xi_n","rho_n","eta_n"]:
    assert np.ptp(autosim_df[col].to_numpy()) < 1e-10

GNDG01_AUTOSIMILAR_NESTED_PASS = True

print(autosim_df.to_string(index=False))
print("GNDG01_AUTOSIMILAR_NESTED_PASS =", GNDG01_AUTOSIMILAR_NESTED_PASS)


 n  a_n       D_n     ell_n       d_n     xi_n  rho_n    eta_n
 0  1.0  1.732051  1.845599  1.732051 1.065557    1.0 1.065557
 1  2.0  3.464102  3.691199  3.464102 1.065557    1.0 1.065557
 2  4.0  6.928203  7.382397  6.928203 1.065557    1.0 1.065557
 3  8.0 13.856406 14.764795 13.856406 1.065557    1.0 1.065557
GNDG01_AUTOSIMILAR_NESTED_PASS = True



# 6. Test D — même trajectoire physique dans des domaines croissants

On garde maintenant **la même trajectoire physique**, donc mêmes extrémités et même longueur, mais on augmente la taille du cube contenant.

Dans ce cas :

\[
\xi_n=\frac{\ell}{d}
\]

doit rester constant, tandis que :

\[
\rho_n=\frac{d}{D_n}
\]

et

\[
\eta_n=\frac{\ell}{D_n}
\]

doivent diminuer lorsque \(D_n\) augmente.

Ici, \(\eta_n\) peut devenir inférieur à 1 sans contradiction, car les extrémités ne sont plus les sommets opposés des cubes plus grands.


In [7]:

# Physical trajectory fixed inside the unit cube.
physical_pts = tapered_helix(1.0, t_dense)

fixed_path_rows = []

for n, a in enumerate(cube_sides):
    assert inside_cube(physical_pts, a)

    d = diagnostics_from_points(physical_pts, a)
    fixed_path_rows.append({
        "n": n,
        "container_side_a_n": a,
        "D_n": d["D"],
        "fixed_d": d["d"],
        "fixed_ell": d["ell"],
        "xi_n": d["xi"],
        "rho_n": d["rho"],
        "eta_n": d["eta"],
    })

fixed_df = pd.DataFrame(fixed_path_rows)

xi_spread = float(np.ptp(fixed_df["xi_n"]))
rho_monotone = bool(np.all(np.diff(fixed_df["rho_n"]) < 0))
eta_monotone = bool(np.all(np.diff(fixed_df["eta_n"]) < 0))

GNDG01_FIXED_PATH_GROWING_DOMAIN_PASS = (
    xi_spread < 1e-12
    and rho_monotone
    and eta_monotone
)

assert GNDG01_FIXED_PATH_GROWING_DOMAIN_PASS

print(fixed_df.to_string(index=False))
print("GNDG01_FIXED_PATH_GROWING_DOMAIN_PASS =", GNDG01_FIXED_PATH_GROWING_DOMAIN_PASS)


 n  container_side_a_n       D_n  fixed_d  fixed_ell     xi_n  rho_n    eta_n
 0                 1.0  1.732051 1.732051   1.845599 1.065557  1.000 1.065557
 1                 2.0  3.464102 1.732051   1.845599 1.065557  0.500 0.532779
 2                 4.0  6.928203 1.732051   1.845599 1.065557  0.250 0.266389
 3                 8.0 13.856406 1.732051   1.845599 1.065557  0.125 0.133195
GNDG01_FIXED_PATH_GROWING_DOMAIN_PASS = True



# 7. Test E — recherche aléatoire de contre-exemple euclidien

On génère des polylignes aléatoires dans le cube unité reliant les deux sommets opposés.

Le test cherche une éventuelle violation numérique de :

\[
\ell\ge D.
\]

Ce test est seulement un contrôle numérique ; la borne est déjà démontrée analytiquement par l'inégalité triangulaire.


In [8]:

rng = np.random.default_rng(20260831)

N_RANDOM = 5000
a = 1.0
D = cube_diagonal(a)

random_lengths = []
violations = 0

for _ in range(N_RANDOM):
    n_internal = int(rng.integers(1, 9))
    pts = np.vstack([
        np.zeros(3),
        rng.uniform(0.0, a, size=(n_internal,3)),
        np.ones(3) * a,
    ])

    ell = arc_length_from_samples(pts)
    random_lengths.append(ell)

    if ell < D - 1e-12:
        violations += 1

random_lengths = np.asarray(random_lengths)
random_min_eta = float(random_lengths.min() / D)

GNDG01_RANDOM_COUNTEREXAMPLE_SEARCH_PASS = (violations == 0)

assert GNDG01_RANDOM_COUNTEREXAMPLE_SEARCH_PASS

random_df = pd.DataFrame([{
    "N_random_paths": N_RANDOM,
    "diagonal_D": D,
    "minimum_random_length": float(random_lengths.min()),
    "minimum_random_eta": random_min_eta,
    "violations": violations,
}])

print(random_df.to_string(index=False))
print("GNDG01_RANDOM_COUNTEREXAMPLE_SEARCH_PASS =", GNDG01_RANDOM_COUNTEREXAMPLE_SEARCH_PASS)


 N_random_paths  diagonal_D  minimum_random_length  minimum_random_eta  violations
           5000    1.732051               1.732688            1.000368           0
GNDG01_RANDOM_COUNTEREXAMPLE_SEARCH_PASS = True



# 8. Test F — invariance sous rotation euclidienne rigide

Avant d'aborder les référentiels relativistes dans une étape future, on vérifie un benchmark plus élémentaire.

Une rotation euclidienne rigide doit conserver :

\[
d,\qquad
\ell,\qquad
\xi.
\]

Si le cube et sa diagonale de référence sont tournés avec le système, elle conserve également :

\[
D,\qquad
\rho,\qquad
\eta.
\]

Ce test ne constitue pas encore un changement de référentiel de Lorentz.


In [9]:

def rotation_matrix(axis, theta):
    axis = np.asarray(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)

    x,y,z = axis
    c = np.cos(theta)
    s = np.sin(theta)
    C = 1-c

    return np.array([
        [c+x*x*C, x*y*C-z*s, x*z*C+y*s],
        [y*x*C+z*s, c+y*y*C, y*z*C-x*s],
        [z*x*C-y*s, z*y*C+x*s, c+z*z*C],
    ])

pts = tapered_helix(1.0, t_dense)
base = diagnostics_from_points(pts, 1.0)

R = rotation_matrix([1.0,2.0,3.0], 0.731)
rot_pts = pts @ R.T

# Rotation about origin preserves distances/arc lengths.
rot_ell = arc_length_from_samples(rot_pts)
rot_d = direct_distance(rot_pts)

rot_xi = rot_ell / rot_d
rot_rho = rot_d / base["D"]
rot_eta = rot_ell / base["D"]

rotation_errors = {
    "ell_error": abs(rot_ell-base["ell"]),
    "d_error": abs(rot_d-base["d"]),
    "xi_error": abs(rot_xi-base["xi"]),
    "rho_error": abs(rot_rho-base["rho"]),
    "eta_error": abs(rot_eta-base["eta"]),
}

GNDG01_EUCLIDEAN_ROTATION_PASS = max(rotation_errors.values()) < 1e-10

assert GNDG01_EUCLIDEAN_ROTATION_PASS

print(rotation_errors)
print("GNDG01_EUCLIDEAN_ROTATION_PASS =", GNDG01_EUCLIDEAN_ROTATION_PASS)


{'ell_error': 0.0, 'd_error': 2.220446049250313e-16, 'xi_error': 2.220446049250313e-16, 'rho_error': np.float64(2.220446049250313e-16), 'eta_error': np.float64(0.0)}
GNDG01_EUCLIDEAN_ROTATION_PASS = True



# 9. Synthèse du diagnostic 0.1

Le notebook teste séparément :

\[
\boxed{
\xi_n=\ell_n/d_n
}
\]

comme diagnostic de non-rectilinéarité,

\[
\boxed{
\rho_n=d_n/D_n
}
\]

comme occupation diagonale relative,

et

\[
\boxed{
\eta_n=\ell_n/D_n=\xi_n\rho_n.
}
\]

## Ce qui peut être conclu dans ce scope

- la borne euclidienne \(\ell\ge d\) est respectée ;
- pour les sommets opposés, \(d=D\), donc \(\eta=\xi\ge1\) ;
- les diagnostics sont numériquement invariants sous reparamétrisation régulière de la même courbe ;
- une famille autosimilaire conserve \((\xi,\rho,\eta)\) ;
- une même trajectoire physique dans des domaines croissants conserve \(\xi\) mais fait décroître \(\rho\) et \(\eta\) ;
- une rotation euclidienne rigide conserve les diagnostics appropriés.

## Ce qui reste ouvert

- référentiels galiléens dynamiques ;
- référentiels lorentziens ;
- géométrie non euclidienne ;
- interprétation observationnelle ;
- relation éventuelle avec les objets dynamiques du noyau GVH.

Aucune de ces propriétés ouvertes n'est supposée ici.


In [10]:

GNDG01_ALL_NUMERIC_TESTS_PASS = all([
    GNDG01_DIAGONAL_BOUND_PASS,
    GNDG01_REPARAMETERIZATION_PASS,
    GNDG01_AUTOSIMILAR_NESTED_PASS,
    GNDG01_FIXED_PATH_GROWING_DOMAIN_PASS,
    GNDG01_RANDOM_COUNTEREXAMPLE_SEARCH_PASS,
    GNDG01_EUCLIDEAN_ROTATION_PASS,
])

GNDG01_GEOMETRIC_DIAGNOSTIC_NUMERICALLY_COHERENT = (
    GNDG01_ALL_NUMERIC_TESTS_PASS
)

GNDG01_DYNAMIC_GVH_LAW_ESTABLISHED = False
GNDG01_LORENTZ_FRAME_BEHAVIOR_ESTABLISHED = False
GNDG01_CURVED_METRIC_EXTENSION_ESTABLISHED = False
GNDG01_OBSERVABLE_PHYSICS_ESTABLISHED = False

assert GNDG01_GEOMETRIC_DIAGNOSTIC_NUMERICALLY_COHERENT
assert not GNDG01_DYNAMIC_GVH_LAW_ESTABLISHED

print("GNDG01_ALL_NUMERIC_TESTS_PASS =", GNDG01_ALL_NUMERIC_TESTS_PASS)
print(
    "GNDG01_GEOMETRIC_DIAGNOSTIC_NUMERICALLY_COHERENT =",
    GNDG01_GEOMETRIC_DIAGNOSTIC_NUMERICALLY_COHERENT
)
print("GNDG01_DYNAMIC_GVH_LAW_ESTABLISHED =", GNDG01_DYNAMIC_GVH_LAW_ESTABLISHED)


GNDG01_ALL_NUMERIC_TESTS_PASS = True
GNDG01_GEOMETRIC_DIAGNOSTIC_NUMERICALLY_COHERENT = True
GNDG01_DYNAMIC_GVH_LAW_ESTABLISHED = False



# 10. Statut quatre niveaux

## Level 1 — hypothèse / définition GVH exploratoire

L'utilisation multi-échelle de

\[
\mathcal E_n=(\xi_n,\rho_n,\eta_n)
\]

comme diagnostic imbriqué est une proposition exploratoire GVH.

## Level 2 — géométrie établie

La borne

\[
\ell\ge d
\]

et la longueur minimale du segment droit sont des résultats euclidiens standard.

## Level 3 — diagnostics numériques

Ce notebook constitue le banc d'essai numérique minimal de la construction.

## Level 4 — dimensions

\[
D,\ d,\ \ell
\]

ont dimension de longueur.

\[
\xi,\ \rho,\ \eta
\]

sont sans dimension.

Aucune échelle SI universelle nouvelle n'est introduite.


In [11]:

FOUR_LEVEL_PROTOCOL_PASS = True
UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK = 0

verdict = {
    "notebook":"GVH_Nested_Diagonal_Geometry_Diagnostic_0.1",
    "status":
        "PASS_EUCLIDEAN_NESTED_DIAGONAL_DIAGNOSTIC_NUMERIC_COHERENCE",
    "scope":
        "exploratory Euclidean nested-cube geometry only",
    "derived":{
        "diagonal_bound_pass":bool(GNDG01_DIAGONAL_BOUND_PASS),
        "reparameterization_pass":bool(GNDG01_REPARAMETERIZATION_PASS),
        "autosimilar_nested_pass":bool(GNDG01_AUTOSIMILAR_NESTED_PASS),
        "fixed_path_growing_domain_pass":bool(
            GNDG01_FIXED_PATH_GROWING_DOMAIN_PASS
        ),
        "random_counterexample_search_pass":bool(
            GNDG01_RANDOM_COUNTEREXAMPLE_SEARCH_PASS
        ),
        "euclidean_rotation_pass":bool(GNDG01_EUCLIDEAN_ROTATION_PASS),
        "geometric_diagnostic_numerically_coherent":bool(
            GNDG01_GEOMETRIC_DIAGNOSTIC_NUMERICALLY_COHERENT
        ),
    },
    "locks":{
        "dynamic_GVH_law_established":False,
        "Lorentz_frame_behavior_established":False,
        "curved_metric_extension_established":False,
        "observable_physics_established":False,
    },
    "protocol":{
        "four_level_protocol_pass":True,
        "universal_theory_selected_SI_scale_rank":0,
    },
    "next_geometry_test":
        "GVH_Nested_Diagonal_Geometry_Diagnostic_0.2_"
        "Reparameterization_Nested_Frames_and_Curved_Metric_Gate",
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)

diag_df.to_csv(export_dir / "gvh_nested_diagonal_0.1_path_diagnostics.csv", index=False)
autosim_df.to_csv(export_dir / "gvh_nested_diagonal_0.1_autosimilar.csv", index=False)
fixed_df.to_csv(export_dir / "gvh_nested_diagonal_0.1_fixed_path_growing_domain.csv", index=False)
random_df.to_csv(export_dir / "gvh_nested_diagonal_0.1_random_search.csv", index=False)

verdict_path = export_dir / "gvh_nested_diagonal_0.1_verdict.json"
verdict_path.write_text(
    json.dumps(verdict, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("STATUS =", verdict["status"])
print("verdict JSON =", verdict_path)


STATUS = PASS_EUCLIDEAN_NESTED_DIAGONAL_DIAGNOSTIC_NUMERIC_COHERENCE
verdict JSON = /content/gvh_exports/gvh_nested_diagonal_0.1_verdict.json
